<a href="https://colab.research.google.com/github/dishanth2k6/Company-Projects/blob/main/Movie.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
import io

# Open the upload widget for the recommendation system dataset
uploaded = files.upload()

# Get the filename dynamically
file_name = list(uploaded.keys())[0]
print(f"\nSuccessfully uploaded: {file_name}")

Saving movielens_100k.csv to movielens_100k.csv

Successfully uploaded: movielens_100k.csv


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_squared_error

print("Recommendation and matrix libraries loaded successfully!")

Recommendation and matrix libraries loaded successfully!


In [4]:
# Load dataset
df = pd.read_csv(io.BytesIO(uploaded[file_name]))

# 1. Handle missing values by replacing them with empty strings
for col in ['genres', 'directors', 'actors', 'title']:
    df[col] = df[col].fillna('')

# 2. Lowercase text entries to normalize tokens
df['genres'] = df['genres'].str.lower()
df['directors'] = df['directors'].str.lower()

# 3. Create a single combined metadata feature string
df['metadata_soup'] = df['title'] + " " + df['genres'] + " " + df['directors']

print("Metadata processing complete. Dataset Shape:", df.shape)
df[['title', 'genres', 'metadata_soup']].head()

Metadata processing complete. Dataset Shape: (1681, 7)


,title,genres,metadata_soup
0,toy story,animation adventure comedy family fantasy,toy story animation adventure comedy family fa...
1,goldeneye,action adventure thriller,goldeneye action adventure thriller martin cam...
2,four rooms,comedy,four rooms comedy allison anders alexandre roc...
3,get shorty,comedy crime thriller,get shorty comedy crime thriller barry sonnenfeld
4,copycat,drama mystery thriller,copycat drama mystery thriller jon amiel


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TF-IDF Vectorizer
tfidf = TfidfVectorizer(stop_words='english', max_features=5000)

# Fit and transform the metadata soup column
tfidf_matrix = tfidf.fit_transform(df['metadata_soup'])

print(f"Metadata converted to TF-IDF matrix! Matrix dimensions: {tfidf_matrix.shape}")

Metadata converted to TF-IDF matrix! Matrix dimensions: (1681, 3586)


In [7]:
# Compute cosine similarity matrix between all vectors
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

print(f"Cosine Similarity Matrix completed! Shape: {cosine_sim.shape}")

Cosine Similarity Matrix completed! Shape: (1681, 1681)


In [8]:
# Create a reverse mapping index for title lookups (and drop duplicates if any exist)
indices = pd.Series(df.index, index=df['title'].str.strip().str.lower()).drop_duplicates()

def get_content_recommendations(title, top_n=5):
    title_clean = str(title).strip().lower()

    if title_clean not in indices:
        return f"Movie title '{title}' not found in the dataset index. Please try another name."

    # Get the row index of the user's movie
    idx = indices[title_clean]

    # Handle cases where multiple index references exist for a single title string name
    if isinstance(idx, pd.Series):
        idx = idx.iloc[0]

    # Grab similarity scores for this movie relative to all others
    sim_scores = list(enumerate(cosine_sim[idx]))

    # Sort the movies based on similarity scores in descending order
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Get scores of the top N most similar movies (skipping the first entry itself)
    sim_scores = sim_scores[1:top_n+1]

    # Get movie indices
    movie_indices = [i[0] for i in sim_scores]
    similarity_weights = [i[1] for i in sim_scores]

    # Return recommendations dataframe
    return pd.DataFrame({
        'Recommended Title': df['title'].iloc[movie_indices].values,
        'Genres': df['genres'].iloc[movie_indices].values,
        'Similarity Score': similarity_weights
    })

# Test the recommendation engine using a popular movie title found in the dataset
sample_title = df['title'].iloc[0] # Typically 'toy story'
print(f"Top 5 Content Recommendations for: '{sample_title}'")
get_content_recommendations(title=sample_title, top_n=5)

Top 5 Content Recommendations for: 'toy story'


,Recommended Title,Genres,Similarity Score
0,neverending story iii the,adventure comedy family fantasy,0.310268
1,aladdin,animation adventure comedy family fantasy musi...,0.280386
2,hercules,animation adventure comedy family fantasy musi...,0.277558
3,dingo,animation adventure comedy family music,0.269340
4,story of xinghua the,,0.256462
